In [7]:
pip uninstall -y scikit-learn scikeras

Found existing installation: scikit-learn 1.5.2
Uninstalling scikit-learn-1.5.2:
  Successfully uninstalled scikit-learn-1.5.2
Found existing installation: scikeras 0.13.0
Uninstalling scikeras-0.13.0:
  Successfully uninstalled scikeras-0.13.0
Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install scikit-learn==1.5.2 scikeras==0.13.0

  Using cached scikit_learn-1.5.2-cp312-cp312-win_amd64.whl.metadata (13 kB)
  Using cached scikeras-0.13.0-py3-none-any.whl.metadata (3.1 kB)
Using cached scikit_learn-1.5.2-cp312-cp312-win_amd64.whl (11.0 MB)
Using cached scikeras-0.13.0-py3-none-any.whl (26 kB)

   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   ---------------------------------------- 0/2 [scikit-learn]
   --------------------------------------


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

In [2]:
# SETUP
os.makedirs("images", exist_ok=True)

# Plot configuration
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
serif_font = "Times New Roman" if "Times New Roman" in available_fonts else \
             "Liberation Serif" if "Liberation Serif" in available_fonts else "DejaVu Serif"

print(f"\n Using font: {serif_font}")

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": [serif_font],
    "font.weight": "bold",
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 14,
    "axes.labelweight": "bold",
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "figure.figsize": (8, 6),
    "savefig.dpi": 600,
    "savefig.format": "eps",
    "savefig.bbox": "tight"
})

def make_bold_ticks(ax):
    """Make all tick labels bold."""
    for label in ax.get_xticklabels():
        label.set_fontweight("bold")
    for label in ax.get_yticklabels():
        label.set_fontweight("bold")

def save_figure(name):
    """Save figure with publication quality."""
    ax = plt.gca()
    for label in ax.get_xticklabels():
        label.set_fontweight("bold")
    for label in ax.get_yticklabels():
        label.set_fontweight("bold")
    plt.savefig(f"images/{name}.eps", format="eps", dpi=600, bbox_inches="tight")
    print(f"images/{name}.eps")

print("Plot configuration complete\n")


 Using font: Times New Roman
Plot configuration complete



In [3]:
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam, SGD, RMSprop


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from scikeras.wrappers import KerasClassifier

In [4]:
#LOAD DATASET
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

class_names = [
    "T-shirt/Top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle Boot"
]

print(f"\nDataset shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  Classes: {len(np.unique(y_train))}")


Dataset shapes:
  X_train: (60000, 28, 28)
  X_test:  (10000, 28, 28)
  Classes: 10


In [5]:
#PREPROCESSING

X_train_flat = X_train.reshape(X_train.shape[0], -1).astype("float32") / 255.0
X_test_flat = X_test.reshape(X_test.shape[0], -1).astype("float32") / 255.0
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

print(f"\nAfter preprocessing:")
print(f"  X_train: {X_train_flat.shape}")
print(f"  X_test:  {X_test_flat.shape}")
print(f"  y_train: {y_train_cat.shape}")
print(f"  y_test:  {y_test_cat.shape}")


After preprocessing:
  X_train: (60000, 784)
  X_test:  (10000, 784)
  y_train: (60000, 10)
  y_test:  (10000, 10)


In [9]:
#SAMPLE IMAGES PLOT

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap="gray")
    ax.set_title(class_names[y_train[i]], fontsize=11, fontweight="bold")
    ax.axis("off")

plt.suptitle("Sample Images from Fashion-MNIST", fontsize=18, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.94])
save_figure("sample_images")
plt.close()

images/sample_images.eps


In [6]:
#CLASS DISTRIBUTION PLOT

class_counts = np.bincount(y_train)
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(class_names, class_counts, edgecolor="black", linewidth=1)
ax.set_title("Class Distribution of Fashion-MNIST Training Dataset", fontweight="bold")
ax.set_xlabel("Class", fontweight="bold")
ax.set_ylabel("Number of Images", fontweight="bold")
plt.xticks(rotation=30, ha="right")
make_bold_ticks(ax)
plt.tight_layout()
save_figure("class_distribution")
plt.close()

images/class_distribution.eps


In [7]:
#BASELINE MODEL

baseline_model = Sequential([
    Dense(128, activation="relu", input_shape=(784,), name="Hidden_Layer_1"),
    Dense(64, activation="relu", name="Hidden_Layer_2"),
    Dense(10, activation="softmax", name="Output_Layer")
])

baseline_model.compile(optimizer=Adam(), loss="categorical_crossentropy", metrics=["accuracy"])

print("\nTraining baseline model (20 epochs)")
history = baseline_model.fit(
    X_train_flat, y_train_cat,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=0
)
print("Baseline training complete")


Training baseline model (20 epochs)
Baseline training complete


In [8]:
#PLOT TRAINING METRICS

epochs = range(1, len(history.history["accuracy"]) + 1)

# Training Accuracy
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(epochs, history.history["accuracy"], linewidth=2.5, label="Training Accuracy")
ax.set_title("Training Accuracy vs Epoch", fontweight="bold")
ax.set_xlabel("Epoch", fontweight="bold")
ax.set_ylabel("Accuracy", fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
make_bold_ticks(ax)
plt.tight_layout()
save_figure("training_accuracy")
plt.close()

# Validation Accuracy
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(epochs, history.history["val_accuracy"], linewidth=2.5, label="Validation Accuracy")
ax.set_title("Validation Accuracy vs Epoch", fontweight="bold")
ax.set_xlabel("Epoch", fontweight="bold")
ax.set_ylabel("Accuracy", fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
make_bold_ticks(ax)
plt.tight_layout()
save_figure("validation_accuracy")
plt.close()

# Training Loss
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(epochs, history.history["loss"], linewidth=2.5, label="Training Loss")
ax.set_title("Training Loss vs Epoch", fontweight="bold")
ax.set_xlabel("Epoch", fontweight="bold")
ax.set_ylabel("Loss", fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
make_bold_ticks(ax)
plt.tight_layout()
save_figure("training_loss")
plt.close()

# Validation Loss
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(epochs, history.history["val_loss"], linewidth=2.5, label="Validation Loss")
ax.set_title("Validation Loss vs Epoch", fontweight="bold")
ax.set_xlabel("Epoch", fontweight="bold")
ax.set_ylabel("Loss", fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
make_bold_ticks(ax)
plt.tight_layout()
save_figure("validation_loss")
plt.close()

print("Training plots saved")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


images/training_accuracy.eps


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


images/validation_accuracy.eps


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


images/training_loss.eps
images/validation_loss.eps
Training plots saved


In [9]:
#BASELINE EVALUATION

y_pred_prob = baseline_model.predict(X_test_flat, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = y_test

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

baseline_metrics = {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

print(f"\nBaseline Model Performance:")
print(f"  Accuracy : {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall   : {recall:.4f}")
print(f"  F1-score : {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, interpolation="nearest")
plt.colorbar(im)
ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right", fontweight="bold")
ax.set_yticklabels(class_names, fontweight="bold")
ax.set_xlabel("Predicted Label", fontweight="bold")
ax.set_ylabel("True Label", fontweight="bold")
ax.set_title("Confusion Matrix (Baseline Model)", fontweight="bold")

threshold = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > threshold else "black",
                fontsize=9, fontweight="bold")

make_bold_ticks(ax)
plt.tight_layout()
save_figure("confusion_matrix")
plt.close()

print("Baseline evaluation complete")


Baseline Model Performance:
  Accuracy : 0.8781
  Precision: 0.8775
  Recall   : 0.8781
  F1-score : 0.8771
images/confusion_matrix.eps
Baseline evaluation complete


In [10]:
# HYPERPARAMETER OPTIMIZATION

def create_mlp(hidden_layers=2, neurons=128, activation="relu",
               learning_rate=0.001, optimizer="adam", dropout_rate=0.0):
    """Build MLP with specified hyperparameters."""
    model = Sequential()
    model.add(Input(shape=(784,)))
    for _ in range(hidden_layers):
        model.add(Dense(neurons, activation=activation))
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))
    model.add(Dense(10, activation="softmax"))

    optimizers_dict = {
        "adam": Adam(learning_rate=learning_rate),
        "sgd": SGD(learning_rate=learning_rate),
        "rmsprop": RMSprop(learning_rate=learning_rate)
    }
    model.compile(optimizer=optimizers_dict[optimizer],
                  loss="categorical_crossentropy", metrics=["accuracy"])
    return model

print("\n Building search space")

param_dist = {
    "model__hidden_layers": [1, 2, 3],
    "model__neurons": [32, 64, 128, 256],
    "model__activation": ["relu", "tanh", "sigmoid"],
    "model__learning_rate": [0.1, 0.01, 0.001],
    "model__optimizer": ["adam", "sgd", "rmsprop"],
    "model__dropout_rate": [0.0, 0.2, 0.5],
    "batch_size": [16, 32, 64, 128],
    "epochs": [10, 20, 30]
}

search_size = 1
for values in param_dist.values():
    search_size *= len(values)

print(f"  Total configurations: {search_size:,}")

# Use subset for speed
USE_DATA_SUBSET = True
SUBSET_SIZE = 30000

if USE_DATA_SUBSET:
    X_train_search = X_train_flat[:SUBSET_SIZE]
    y_train_search = y_train_cat[:SUBSET_SIZE]
    print(f"  Using subset: {SUBSET_SIZE:,} samples")
else:
    X_train_search = X_train_flat
    y_train_search = y_train_cat
    print(f"  Using full dataset: 60,000 samples")

# Create wrapper
mlp_classifier = KerasClassifier(model=create_mlp, verbose=0)

# Create search
random_search = RandomizedSearchCV(
    estimator=mlp_classifier,
    param_distributions=param_dist,
    n_iter=15,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=2,  # Safe for local machine
    verbose=1,
    return_train_score=True
)

print("\n Starting RandomizedSearchCV")
print(f"  Iterations: 15")
print(f"  CV folds: 5")
print(f"  Total trainings: 75")

t_start = time.time()
random_search.fit(X_train_search, y_train_search)
t_search = time.time() - t_start


 Building search space
  Total configurations: 11,664
  Using subset: 30,000 samples

 Starting RandomizedSearchCV
  Iterations: 15
  CV folds: 5
  Total trainings: 75
Fitting 5 folds for each of 15 candidates, totalling 75 fits


In [11]:
# ANALYSIS

best_params = random_search.best_params_
print(f"\nCross-Validation Accuracy: {random_search.best_score_:.4f}")
print("\nBest Hyperparameters:")
for key in sorted(best_params.keys()):
    clean_key = key.replace("model__", "")
    print(f"  {clean_key:25} : {best_params[key]}")

results_df = pd.DataFrame(random_search.cv_results_).sort_values(by="rank_test_score")
results_df.to_csv("hyperparameter_search_results.csv", index=False)
print(f"\n✓ Saved: hyperparameter_search_results.csv")

# Plot search results
top_n = 10
top_results = results_df.nsmallest(top_n, "rank_test_score")
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(top_results))
colors = plt.cm.viridis(np.linspace(0, 1, len(top_results)))
bars = ax.bar(x_pos, top_results["mean_test_score"].values, color=colors, edgecolor="black", linewidth=1.5)
ax.errorbar(x_pos, top_results["mean_test_score"].values, yerr=top_results["std_test_score"].values,
            fmt="none", ecolor="black", capsize=5, linewidth=1.5)
ax.set_xlabel("Configuration Rank", fontweight="bold", fontsize=14)
ax.set_ylabel("Mean Cross-Validation Accuracy", fontweight="bold", fontsize=14)
ax.set_title("Top 10 Hyperparameter Configurations", fontweight="bold", fontsize=16)
ax.set_xticks(x_pos)
ax.set_xticklabels([f"#{i+1}" for i in range(len(top_results))], fontweight="bold")
ax.grid(axis="y", alpha=0.3, linestyle="--")
make_bold_ticks(ax)
plt.tight_layout()
save_figure("hyperparameter_search_results")
plt.close()

print("Hyperparameter search plot saved")


Cross-Validation Accuracy: 0.8736

Best Hyperparameters:
  batch_size                : 16
  epochs                    : 20
  activation                : sigmoid
  dropout_rate              : 0.0
  hidden_layers             : 2
  learning_rate             : 0.001
  neurons                   : 64
  optimizer                 : rmsprop

✓ Saved: hyperparameter_search_results.csv


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


images/hyperparameter_search_results.eps
Hyperparameter search plot saved


In [12]:
# RETRAIN OPTIMIZED MODEL

best_params_dict = best_params.copy()
model_params = {k.replace("model__", ""): v for k, v in best_params_dict.items() if k.startswith("model__")}
fit_params = {k: v for k, v in best_params_dict.items() if not k.startswith("model__")}

print(f"\nTraining optimized model on full 60,000 samples...")

optimized_model = create_mlp(**model_params)
history_optimized = optimized_model.fit(
    X_train_flat, y_train_cat,
    batch_size=int(fit_params.get("batch_size", 32)),
    epochs=int(fit_params.get("epochs", 20)),
    validation_split=0.2,
    verbose=0
)

print("Optimized model training complete")


Training optimized model on full 60,000 samples...
Optimized model training complete


In [13]:
# EVALUATE OPTIMIZED MODEL

y_pred_prob_opt = optimized_model.predict(X_test_flat, verbose=0)
y_pred_opt = np.argmax(y_pred_prob_opt, axis=1)

accuracy_opt = accuracy_score(y_test, y_pred_opt)
precision_opt = precision_score(y_test, y_pred_opt, average="weighted")
recall_opt = recall_score(y_test, y_pred_opt, average="weighted")
f1_opt = f1_score(y_test, y_pred_opt, average="weighted")

optimized_metrics = {"accuracy": accuracy_opt, "precision": precision_opt, "recall": recall_opt, "f1": f1_opt}

print(f"\nOptimized Model Performance:")
print(f"  Accuracy : {accuracy_opt:.4f}")
print(f"  Precision: {precision_opt:.4f}")
print(f"  Recall   : {recall_opt:.4f}")
print(f"  F1-score : {f1_opt:.4f}")


Optimized Model Performance:
  Accuracy : 0.8712
  Precision: 0.8738
  Recall   : 0.8712
  F1-score : 0.8707


In [14]:
# COMPARISON

comparison_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Baseline": [baseline_metrics["accuracy"], baseline_metrics["precision"],
                 baseline_metrics["recall"], baseline_metrics["f1"]],
    "Optimized": [optimized_metrics["accuracy"], optimized_metrics["precision"],
                  optimized_metrics["recall"], optimized_metrics["f1"]]
})

comparison_df["Improvement"] = comparison_df["Optimized"] - comparison_df["Baseline"]
comparison_df["% Improvement"] = (comparison_df["Improvement"] / comparison_df["Baseline"] * 100).round(2)

print("\nPerformance Comparison:")
print(comparison_df.to_string(index=False))

comparison_df.to_csv("baseline_vs_optimized_comparison.csv", index=False)
print(f"\n✓ Saved: baseline_vs_optimized_comparison.csv")

# Plot comparison
fig, ax = plt.subplots(figsize=(9, 6))
metrics = comparison_df["Metric"].values
x = np.arange(len(metrics))
width = 0.35
ax.bar(x - width/2, comparison_df["Baseline"].values, width, label="Baseline", color="#1f77b4", edgecolor="black", linewidth=1.5)
ax.bar(x + width/2, comparison_df["Optimized"].values, width, label="Optimized", color="#ff7f0e", edgecolor="black", linewidth=1.5)
ax.set_ylabel("Score", fontweight="bold", fontsize=14)
ax.set_title("Baseline vs Optimized Model Performance", fontweight="bold", fontsize=16)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontweight="bold")
ax.legend(fontsize=12, loc="lower right")
ax.set_ylim([0.85, 0.91])
ax.grid(axis="y", alpha=0.3, linestyle="--")
make_bold_ticks(ax)
plt.tight_layout()
save_figure("baseline_vs_optimized_accuracy")
plt.close()

print("Comparison plot saved")

# Optimized confusion matrix
cm_opt = confusion_matrix(y_test, y_pred_opt)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_opt, interpolation="nearest", cmap="Blues")
plt.colorbar(im, ax=ax, label="Count")
ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right", fontweight="bold")
ax.set_yticklabels(class_names, fontweight="bold")
ax.set_xlabel("Predicted Label", fontweight="bold", fontsize=14)
ax.set_ylabel("True Label", fontweight="bold", fontsize=14)
ax.set_title("Confusion Matrix (Optimized Model)", fontweight="bold", fontsize=16)
threshold = cm_opt.max() / 2
for i in range(cm_opt.shape[0]):
    for j in range(cm_opt.shape[1]):
        ax.text(j, i, str(cm_opt[i, j]), ha="center", va="center",
                color="white" if cm_opt[i, j] > threshold else "black",
                fontsize=9, fontweight="bold")
make_bold_ticks(ax)
plt.tight_layout()
save_figure("confusion_matrix_optimized")
plt.close()

print("Optimized confusion matrix saved")


Performance Comparison:
   Metric  Baseline  Optimized  Improvement  % Improvement
 Accuracy  0.878100   0.871200    -0.006900          -0.79
Precision  0.877524   0.873781    -0.003743          -0.43
   Recall  0.878100   0.871200    -0.006900          -0.79
 F1-Score  0.877090   0.870722    -0.006368          -0.73

✓ Saved: baseline_vs_optimized_comparison.csv


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


images/baseline_vs_optimized_accuracy.eps
Comparison plot saved
images/confusion_matrix_optimized.eps
Optimized confusion matrix saved


In [16]:
baseline_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ Hidden_Layer_1 (Dense)               │ (None, 128)                 │         100,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Hidden_Layer_2 (Dense)               │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Output_Layer (Dense)                 │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 328,160 (1.25 MB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 218,774 (854.59 KB)

In [17]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true, y_pred,
    target_names=class_names,
    digits=4
))

              precision    recall  f1-score   support

 T-shirt/Top     0.8781    0.7710    0.8211      1000
     Trouser     0.9676    0.9850    0.9762      1000
    Pullover     0.7887    0.7950    0.7918      1000
       Dress     0.8763    0.9000    0.8880      1000
        Coat     0.7739    0.8490    0.8097      1000
      Sandal     0.9753    0.9490    0.9620      1000
       Shirt     0.7027    0.6500    0.6753      1000
     Sneaker     0.9304    0.9490    0.9396      1000
         Bag     0.9425    0.9830    0.9623      1000
  Ankle Boot     0.9397    0.9500    0.9448      1000

    accuracy                         0.8781     10000
   macro avg     0.8775    0.8781    0.8771     10000
weighted avg     0.8775    0.8781    0.8771     10000



In [15]:
# FINAL SUMMARY

print(f"\n1. METHOD: RandomizedSearchCV with SciKeras")
print(f"   Search space: {search_size:,} possible configurations")
print(f"   Configurations tried: 15")
print(f"   CV folds: 5")
print(f"   Total trainings: 75")
print(f"   Search time: {t_search/60:.1f} minutes")

print(f"\n2. BEST HYPERPARAMETERS:")
for key, value in sorted(best_params.items()):
    clean_key = key.replace("model__", "")
    print(f"   {clean_key:25} : {value}")

print(f"\n3. CROSS-VALIDATION ACCURACY: {random_search.best_score_:.4f}")

print(f"\n4. PERFORMANCE IMPROVEMENT:")
for _, row in comparison_df.iterrows():
    metric = row["Metric"]
    baseline = row["Baseline"]
    optimized = row["Optimized"]
    improvement = row["Improvement"]
    pct = row["% Improvement"]
    symbol = "Yes" if improvement >= 0 else "No"
    print(f"   {symbol} {metric:10} : {baseline:.4f} → {optimized:.4f} ({pct:+.2f}%)")

print(f"\n5. GENERATED FILES:")
print(f"   Plots: images/ folder (10 EPS files, 600 DPI)")
print(f"   Data: hyperparameter_search_results.csv")
print(f"   Data: baseline_vs_optimized_comparison.csv")


1. METHOD: RandomizedSearchCV with SciKeras
   Search space: 11,664 possible configurations
   Configurations tried: 15
   CV folds: 5
   Total trainings: 75
   Search time: 38.3 minutes

2. BEST HYPERPARAMETERS:
   batch_size                : 16
   epochs                    : 20
   activation                : sigmoid
   dropout_rate              : 0.0
   hidden_layers             : 2
   learning_rate             : 0.001
   neurons                   : 64
   optimizer                 : rmsprop

3. CROSS-VALIDATION ACCURACY: 0.8736

4. PERFORMANCE IMPROVEMENT:
   No Accuracy   : 0.8781 → 0.8712 (-0.79%)
   No Precision  : 0.8775 → 0.8738 (-0.43%)
   No Recall     : 0.8781 → 0.8712 (-0.79%)
   No F1-Score   : 0.8771 → 0.8707 (-0.73%)

5. GENERATED FILES:
   Plots: images/ folder (10 EPS files, 600 DPI)
   Data: hyperparameter_search_results.csv
   Data: baseline_vs_optimized_comparison.csv
